# YOTO pipeline: preprocess, Zuna, and LaBraM training


This notebook now embeds the helper logic that was previously hidden in the `scripts/` helpers. Each
section defines the functions that build manifests, preprocess YOTO tones, export FIF files, run Zuna,
and train LaBraM so you can inspect intermediate data without shelling out to separate scripts.


In [ ]:

from __future__ import annotations

import json
import logging
import os
import re
import sys
import tempfile
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset

LOG = logging.getLogger("yoto_pipeline")
LOG.setLevel(logging.INFO)
if not LOG.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
    LOG.addHandler(handler)

try:
    import zuna
except ModuleNotFoundError:  # pragma: no cover
    zuna = None
    LOG.warning("ZUNA is not installed; augmentation stages will be skipped.")

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

VENDOR_LABRAM = ROOT / "vendor/LaBraM"
if str(VENDOR_LABRAM) not in sys.path:
    sys.path.insert(0, str(VENDOR_LABRAM))

try:
    from modeling_finetune import labram_base_patch200_200  # type: ignore
except ModuleNotFoundError:  # pragma: no cover
    labram_base_patch200_200 = None
    LOG.warning("LaBraM vendor repo not found; training stage requires vendor/LaBraM.")

DATA_ROOT = ROOT / "data"
RAW_ROOT = DATA_ROOT / "raw_samples"
MANIFEST_DIR = DATA_ROOT / "manifests"
PROCESSED_DIR = DATA_ROOT / "processed"
EPOCH_DIR = PROCESSED_DIR / "epochs_yoto_tones"
FIF_DIR = PROCESSED_DIR / "fif_for_zuna"
CONFIG_PREPROC = ROOT / "configs/yoto_preprocessing.yaml"
CONFIG_EVENTS = ROOT / "configs/yoto_events.yaml"
UNIFIED_MANIFEST_JSON = MANIFEST_DIR / "unified_manifest.json"
UNIFIED_MANIFEST_CSV = MANIFEST_DIR / "unified_manifest.csv"
EPOCH_INDEX_PATH = MANIFEST_DIR / "epoch_index_yoto_tones.csv"
RUNS_DIR = ROOT / "runs"
ZUNA_WORK_DIR = RUNS_DIR / "zuna_yoto"
LABRAM_METRICS_PATH = RUNS_DIR / "labram_metrics.json"
YOTO_DATASET_ID = "ds005815"


In [ ]:

EEG_EXTS = {".vhdr", ".edf", ".bdf", ".set", ".fif", ".mat", ".eeg", ".vmrk"}

def infer_subject(path: Path) -> str:
    match = re.search(r"(sub-[A-Za-z0-9]+)", str(path))
    return match.group(1) if match else "unknown_subject"


def infer_modality(path: Path) -> str:
    name = path.name.lower()
    if "audiovisual" in name or name.startswith("av"):
        return "audiovisual"
    if any(k in name for k in ["audio", "auditory", "oddball", "tone", "vowel"]):
        return "auditory"
    if any(k in name for k in ["visual", "color", "colour", "rgb", "gabor", "face"]):
        return "visual"
    return "unknown"


def infer_task_label(path: Path) -> str:
    name = path.name.lower()
    if "rest" in name:
        return "rest"
    if "oddball" in name:
        return "oddball"
    if "task" in name:
        return "task"
    if "rgb" in name:
        return "rgb"
    if "color" in name or "colour" in name:
        return "color"
    return "unknown"


def build_unified_manifest(force: bool = False) -> pd.DataFrame:
    if not RAW_ROOT.exists():
        LOG.warning("Raw samples directory %s does not exist yet.", RAW_ROOT)
    if not force and UNIFIED_MANIFEST_CSV.exists():
        LOG.info("Loading cached manifest from %s", UNIFIED_MANIFEST_CSV)
        return pd.read_csv(UNIFIED_MANIFEST_CSV)

    rows: list[dict[str, Any]] = []
    for path in RAW_ROOT.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in EEG_EXTS:
            continue
        rel = path.relative_to(RAW_ROOT)
        dataset = rel.parts[0] if rel.parts else "unknown"
        rows.append({
            "dataset_id": dataset,
            "subject_id": infer_subject(path),
            "file_path": str(path),
            "file_ext": path.suffix.lower(),
            "modality": infer_modality(path),
            "task_label": infer_task_label(path),
            "has_events_sidecar": path.with_suffix(".tsv").exists()
            or path.with_suffix(".json").exists(),
        })

    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(UNIFIED_MANIFEST_CSV, index=False)
    UNIFIED_MANIFEST_JSON.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    LOG.info("Wrote %d manifest rows to %s", len(rows), UNIFIED_MANIFEST_JSON)
    return df


In [ ]:

manifest_df = build_unified_manifest()
manifest_df.head()


In [ ]:

def load_config(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    with path.open() as handle:
        return yaml.safe_load(handle) or {}


def load_events_tsv(vhdr_path: Path) -> pd.DataFrame | None:
    stem = vhdr_path.stem.replace("_eeg", "")
    tsv = vhdr_path.parent / (stem + "_events.tsv")
    if not tsv.exists():
        tsv = vhdr_path.with_suffix(".tsv")
    if not tsv.exists():
        return None
    return pd.read_csv(tsv, sep="	")


def get_event_mapping(config_events: dict[str, Any]) -> dict[int | str, str]:
    out: dict[int | str, str] = {}
    for k, v in config_events.get("event_value_to_stimulus", {}).items():
        try:
            out[int(k)] = str(v)
        except ValueError:
            out[str(k)] = str(v)
    for k, v in config_events.get("trial_type_to_stimulus", {}).items():
        out[str(k)] = str(v)
    return out


def load_raw(path: Path) -> mne.io.Raw | None:
    if path.suffix.lower() != ".vhdr":
        return None
    try:
        return mne.io.read_raw_brainvision(path, preload=True, verbose=False)
    except Exception as exc:  # noqa: BLE001
        LOG.warning("Failed to load %s (%s)", path, exc)
        return None


def apply_causal_fir(raw: mne.io.Raw, cfg: dict[str, Any]) -> None:
    pre = cfg.get("preprocessing", cfg)
    l_freq, h_freq = pre.get("bandpass_hz", [1, 50])
    buf_sec = pre.get("filter_buffer_seconds", 30)
    sfreq = raw.info["sfreq"]
    filter_length = max(int(buf_sec * sfreq), 3500)
    filtered = mne.filter.filter_data(
        raw.get_data(),
        sfreq,
        l_freq,
        h_freq,
        filter_length=filter_length,
        phase="minimum",
        verbose=False,
    )
    raw._data[:] = filtered


def apply_asr(raw: mne.io.Raw, cfg: dict[str, Any]) -> None:
    pre = cfg.get("preprocessing", cfg)
    if not pre.get("asr_enabled", True):
        return
    try:
        import asrpy
    except ImportError:
        LOG.warning("asrpy not installed; skipping ASR")
        return
    thresh = pre.get("asr_threshold", 20)
    data = raw.get_data()
    asr = asrpy.ASR(sfreq=raw.info["sfreq"], cutoff=thresh)
    asr.fit(raw.get_data())
    cleaned = asr.transform(data)
    raw._data[:] = cleaned


def apply_ica_iclabel(raw: mne.io.Raw, cfg: dict[str, Any]) -> None:
    pre = cfg.get("preprocessing", cfg)
    if not pre.get("ica_enabled", True):
        return
    n_comp = pre.get("ica_max_pca_components")
    if n_comp is None:
        n_comp = min(raw.info["nchan"], 64) - 1
    ica = mne.preprocessing.ICA(n_components=n_comp, random_state=42, method="fastica")
    ica.fit(raw, verbose=False)
    thresh = pre.get("iclabel_reject_threshold", 0.8)
    exclude: list[int] = []
    try:
        from mne_icalabel import label_components

        ic_labels = label_components(raw, ica, method="iclabel")
        for idx, probs in enumerate(ic_labels.get("y_pred_proba", [])):
            if isinstance(probs, (list, np.ndarray)) and len(probs) >= 2:
                if probs[1] > thresh or (len(probs) > 2 and probs[2] > thresh):
                    exclude.append(idx)
    except ImportError:
        LOG.warning("mne_icalabel is not installed; skipping ICA labeling")
    ica.exclude = exclude
    cleaned = ica.apply(raw.copy(), verbose=False)
    raw._data[:] = cleaned._data


def preprocess_chain(raw: mne.io.Raw, cfg: dict[str, Any]) -> None:
    apply_causal_fir(raw, cfg)
    pre = cfg.get("preprocessing", cfg)
    sfreq = pre.get("resample_hz", 250)
    raw.resample(sfreq, verbose=False)
    apply_asr(raw, cfg)
    apply_ica_iclabel(raw, cfg)
    if pre.get("rereference") == "average":
        raw.set_eeg_reference("average", verbose=False)


def events_to_epochs(
    raw: mne.io.Raw,
    events_df: pd.DataFrame,
    event_mapping: dict[int | str, str],
    tmin: float,
    tmax: float,
    stim_col: str = "value",
    trial_type_col: str = "trial_type",
    onset_col: str = "onset",
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    data = raw.get_data()
    sfreq = raw.info["sfreq"]
    n_samp_pre = int(round(-tmin * sfreq))
    n_samp_post = int(round(tmax * sfreq))
    n_samp = n_samp_pre + n_samp_post
    n_ch = data.shape[0]
    stimulus_ids: list[str] = []
    valid_starts: list[int] = []
    for _, row in events_df.iterrows():
        stim_id: str | None = None
        if stim_col in row.index and pd.notna(row[stim_col]):
            try:
                val = int(float(row[stim_col]))
                stim_id = event_mapping.get(val)
            except (ValueError, TypeError):
                pass
        if stim_id is None and trial_type_col in row.index:
            stim_id = event_mapping.get(str(row[trial_type_col]).strip())
        if stim_id not in ("tone_C", "tone_D", "tone_E"):
            continue
        onset = float(row.get(onset_col, 0.0))
        onset_samp = int(round(onset * sfreq))
        start = onset_samp - n_samp_pre
        end = onset_samp + n_samp_post
        if start < 0 or end > data.shape[1]:
            continue
        stimulus_ids.append(stim_id)
        valid_starts.append(start)
    if not valid_starts:
        return (
            np.empty((0, n_ch, n_samp), dtype=np.float32),
            np.array([], dtype=int),
            [],
        )
    epochs = np.zeros((len(valid_starts), n_ch, n_samp), dtype=np.float32)
    for idx, start in enumerate(valid_starts):
        epochs[idx] = data[:, start : start + n_samp]
    return epochs, np.array(valid_starts), stimulus_ids


def _filter_manifest_for_yoto(
    manifest: pd.DataFrame,
    dataset_id: str,
    task_contains: str,
) -> list[Path]:
    df = manifest.copy()
    df = df[df["file_ext"] == ".vhdr"]
    df = df[df["dataset_id"] == dataset_id]
    df = df[df["file_path"].str.contains(task_contains, case=False, na=False)]
    return [Path(p) for p in df["file_path"].unique()]


def preprocess_yoto_tones(
    manifest: pd.DataFrame | None = None,
    dataset_id: str = YOTO_DATASET_ID,
    task_contains: str = "task-task",
    skip_asr: bool = False,
    skip_ica: bool = False,
    epoch_dir: Path = EPOCH_DIR,
    epoch_index_path: Path = EPOCH_INDEX_PATH,
) -> pd.DataFrame:
    cfg = load_config(CONFIG_PREPROC)
    cfg = dict(cfg)
    cfg.setdefault("preprocessing", {})
    if skip_asr:
        cfg["preprocessing"]["asr_enabled"] = False
    if skip_ica:
        cfg["preprocessing"]["ica_enabled"] = False
    epoch_cfg = cfg.get("epoch", {})
    tmin = float(epoch_cfg.get("tmin", -0.2))
    tmax = float(epoch_cfg.get("tmax", 0.8))

    cfg_events = load_config(CONFIG_EVENTS)
    event_mapping = get_event_mapping(cfg_events)
    if not event_mapping:
        LOG.warning("No event mapping found in %s", CONFIG_EVENTS)

    if manifest is None:
        if UNIFIED_MANIFEST_CSV.exists():
            manifest = pd.read_csv(UNIFIED_MANIFEST_CSV)
        else:
            manifest = pd.DataFrame(columns=["file_path"])
    vhdr_paths = _filter_manifest_for_yoto(manifest, dataset_id, task_contains)
    if not vhdr_paths:
        fallback = list((RAW_ROOT / dataset_id).rglob(f"*{task_contains}*.vhdr"))
        vhdr_paths = [p for p in fallback if p.is_file()]

    epoch_dir.mkdir(parents=True, exist_ok=True)
    index_rows: list[dict[str, Any]] = []
    for vhdr in sorted(vhdr_paths):
        raw = load_raw(vhdr)
        if raw is None:
            continue
        preprocess_chain(raw, cfg)
        events_df = load_events_tsv(vhdr)
        if events_df is None:
            continue
        epochs, _, stimulus_ids = events_to_epochs(raw, events_df, event_mapping, tmin, tmax)
        if epochs.size == 0:
            continue
        subject_id = next((part for part in vhdr.parts if part.startswith("sub-")), "unknown_subject")
        stem = f"{dataset_id}__{subject_id}__{vhdr.stem}"
        out_path = epoch_dir / f"{stem}.npy"
        np.save(out_path, epochs)
        for epoch_idx, stimulus_id in enumerate(stimulus_ids):
            index_rows.append(
                {
                    "dataset_id": dataset_id,
                    "subject_id": subject_id,
                    "source_file": str(vhdr),
                    "epochs_file": str(out_path),
                    "epoch_idx": epoch_idx,
                    "stimulus_id": stimulus_id,
                    "n_channels": int(epochs.shape[1]),
                    "n_samples": int(epochs.shape[2]),
                }
            )
    df = pd.DataFrame(index_rows)
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    if not df.empty:
        df.to_csv(epoch_index_path, index=False)
        (epoch_index_path.with_suffix(".json")).write_text(json.dumps(index_rows, indent=2), encoding="utf-8")
        LOG.info("Saved %d tone epochs to %s", len(index_rows), epoch_index_path)
    else:
        LOG.warning("No tone epochs produced; check event mapping and raw files.")
    return df


In [ ]:

epoch_index_df = preprocess_yoto_tones(manifest_df, skip_asr=True, skip_ica=True)
epoch_index_df.head()


In [ ]:

def export_preprocessed_yoto_to_fif(
    manifest: pd.DataFrame | None = None,
    dataset_id: str = YOTO_DATASET_ID,
    task_contains: str = "task-task",
    skip_asr: bool = False,
    skip_ica: bool = False,
    out_dir: Path = FIF_DIR,
) -> list[Path]:
    cfg = load_config(CONFIG_PREPROC)
    cfg = dict(cfg)
    cfg.setdefault("preprocessing", {})
    if skip_asr:
        cfg["preprocessing"]["asr_enabled"] = False
    if skip_ica:
        cfg["preprocessing"]["ica_enabled"] = False

    if manifest is None and UNIFIED_MANIFEST_CSV.exists():
        manifest = pd.read_csv(UNIFIED_MANIFEST_CSV)
    vhdr_paths = _filter_manifest_for_yoto(manifest, dataset_id, task_contains)
    if not vhdr_paths:
        vhdr_paths = [p for p in (RAW_ROOT / dataset_id).rglob("*task-task*.vhdr") if p.is_file()]
    out_dir.mkdir(parents=True, exist_ok=True)
    saved: list[Path] = []
    for vhdr in sorted(vhdr_paths):
        raw = load_raw(vhdr)
        if raw is None:
            continue
        preprocess_chain(raw, cfg)
        subject_id = next((part for part in vhdr.parts if part.startswith("sub-")), "unknown_subject")
        stem = f"{dataset_id}__{subject_id}__{vhdr.stem}"
        out_path = out_dir / f"{stem}.fif"
        raw.save(out_path, overwrite=True, verbose=False)
        saved.append(out_path)
        LOG.info("Exported FIF %s", out_path)
    return saved


In [ ]:

fif_paths = export_preprocessed_yoto_to_fif(manifest_df, skip_asr=True, skip_ica=True)
fif_paths[:5]


In [ ]:

def run_zuna_augmentation(
    input_dir: Path = FIF_DIR,
    work_dir: Path = ZUNA_WORK_DIR,
    subject_filter: str = "",
    gpu_device: str = "0",
    tokens_per_batch: int = 2048,
    diffusion_steps: int = 8,
) -> dict[str, Any]:
    if zuna is None:
        LOG.warning("ZUNA library missing; cannot run augmentation.")
        return {"status": "missing_zuna"}
    work_dir = Path(work_dir)
    d1 = work_dir / "1_fif_filter"
    d2 = work_dir / "2_pt_input"
    d3 = work_dir / "3_pt_output"
    d4 = work_dir / "4_fif_output"
    fig_dir = work_dir / "FIGURES"
    for d in (d1, d2, d3, d4, fig_dir):
        d.mkdir(parents=True, exist_ok=True)
    input_path = Path(input_dir)
    zuna_input_dir = input_path
    if subject_filter:
        matches = sorted(input_path.glob(f"*{subject_filter}*.fif"))
        if not matches:
            raise ValueError(f"No FIF files matched subject filter {subject_filter}")
        temp_dir = Path(tempfile.mkdtemp(prefix="zuna_input_"))
        for src in matches:
            (temp_dir / src.name).symlink_to(src.resolve())
        LOG.info("Subject filter %s selected %d files", subject_filter, len(matches))
        zuna_input_dir = temp_dir
    zuna.preprocessing(
        input_dir=str(zuna_input_dir),
        output_dir=str(d2),
        apply_notch_filter=False,
        apply_highpass_filter=True,
        apply_average_reference=True,
        target_channel_count=["AF3", "AF4", "F1", "F2", "C3", "C4", "P3", "P4"],
        bad_channels=[],
        preprocessed_fif_dir=str(d1),
    )
    zuna.inference(
        input_dir=str(d2),
        output_dir=str(d3),
        gpu_device=gpu_device,
        tokens_per_batch=tokens_per_batch,
        data_norm=10.0,
        diffusion_cfg=1.0,
        diffusion_sample_steps=diffusion_steps,
        plot_eeg_signal_samples=False,
        inference_figures_dir=str(fig_dir),
    )
    zuna.pt_to_fif(input_dir=str(d3), output_dir=str(d4))
    summary = {
        "status": "completed",
        "output_dir": str(d4),
        "diffusion_steps": diffusion_steps,
        "preprocessed_dir": str(d2),
    }
    LOG.info("ZUNA outputs written to %s", d4)
    return summary


In [ ]:

RUN_ZUNA = False
zuna_summary = {}
if RUN_ZUNA:
    zuna_summary = run_zuna_augmentation()
else:
    LOG.info("ZUNA augmentation skipped; set RUN_ZUNA=True to execute.")
zuna_summary


In [ ]:

class LabramEpochDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


def prepare_for_labram(epochs: np.ndarray, patch_size: int = 200, n_patches: int = 6) -> np.ndarray:
    target_len = patch_size * n_patches
    if epochs.shape[2] < target_len:
        pad = target_len - epochs.shape[2]
        epochs = np.pad(epochs, ((0, 0), (0, 0), (0, pad)))
    elif epochs.shape[2] > target_len:
        epochs = epochs[:, :, :target_len]
    epochs = epochs.mean(axis=1, keepdims=True)
    epochs = epochs.reshape(epochs.shape[0], 1, n_patches, patch_size)
    return epochs.astype(np.float32, copy=False)


def _load_data_long_format(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_list, y_list, g_list = [], [], []
    for epochs_file, group in df.groupby("epochs_file"):
        arr = np.load(epochs_file)
        for row in group.itertuples(index=False):
            epoch_idx = int(getattr(row, "epoch_idx", 0))
            stim_id = getattr(row, "stimulus_id", None)
            if stim_id is None:
                continue
            x = arr[epoch_idx : epoch_idx + 1]
            x = prepare_for_labram(x)
            X_list.append(x)
            y_list.append(stim_id)
            subj = getattr(row, "subject_id", "unknown_subject")
            ds = getattr(row, "dataset_id", "unknown")
            g_list.append(f"{ds}::{subj}")
    X = np.concatenate(X_list, axis=0) if X_list else np.empty((0, 1, 6, 200), dtype=np.float32)
    y = np.array(y_list)
    g = np.array(g_list)
    return X, y, g


def _load_data_per_recording(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_all, y_all, g_all = [], [], []
    for row in df.itertuples(index=False):
        epochs = np.load(row.epochs_file)
        epochs = prepare_for_labram(epochs)
        X_all.append(epochs)
        y_all.extend([row.modality] * len(epochs))
        g_all.extend([f"{row.dataset_id}::{row.subject_id}"] * len(epochs))
    X = np.concatenate(X_all, axis=0)
    y = np.array(y_all)
    g = np.array(g_all)
    return X, y, g


def load_labram_epoch_data(epoch_index_path: Path = EPOCH_INDEX_PATH) -> tuple[np.ndarray, np.ndarray, np.ndarray, LabelEncoder]:
    df = pd.read_csv(epoch_index_path)
    if "stimulus_id" in df.columns and "epoch_idx" in df.columns:
        X, y, groups = _load_data_long_format(df)
    else:
        X, y, groups = _load_data_per_recording(df)
    label_encoder = LabelEncoder()
    y_enc = label_encoder.fit_transform(y)
    return X, y_enc, groups, label_encoder


def train_labram_model(
    epoch_index_path: Path = EPOCH_INDEX_PATH,
    device: str = "cpu",
    batch_size: int = 16,
    epochs: int = 3,
    lr: float = 1e-4,
    checkpoint: str = "",
) -> dict[str, Any]:
    if labram_base_patch200_200 is None:
        raise RuntimeError("LaBraM vendor model is not available; supply vendor/LaBraM.")
    X, y_enc, groups, label_encoder = load_labram_epoch_data(epoch_index_path)
    if len(y_enc) < 2:
        raise ValueError("Not enough samples to train LaBraM")
    if len(set(groups)) < 2:
        splitter = ShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
        train_idx, test_idx = next(splitter.split(X, y_enc))
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
        train_idx, test_idx = next(splitter.split(X, y_enc, groups))
    train_ds = LabramEpochDataset(X[train_idx], y_enc[train_idx])
    test_ds = LabramEpochDataset(X[test_idx], y_enc[test_idx])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    model = labram_base_patch200_200(
        num_classes=len(label_encoder.classes_),
        EEG_size=1200,
        init_values=0.1,
    ).to(device)
    if checkpoint:
        ckpt = torch.load(checkpoint, map_location="cpu")
        state = ckpt.get("model", ckpt)
        current = model.state_dict()
        filtered = {k: v for k, v in state.items() if k in current and v.shape == current[k].shape}
        model.load_state_dict(filtered, strict=False)
        LOG.info("Loaded checkpoint with %d matching keys", len(filtered))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = torch.nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb, input_chans=[0, 1])
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb, input_chans=[0, 1]).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += len(yb)
    metrics = {
        "accuracy": float(correct / max(total, 1)),
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "label_classes": label_encoder.classes_.tolist(),
        "input_shape": list(X.shape[1:]),
        "device": device,
        "epochs": epochs,
        "batch_size": batch_size,
    }
    LABRAM_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    LABRAM_METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    LOG.info("Saved LaBraM metrics to %s", LABRAM_METRICS_PATH)
    return metrics


In [ ]:

RUN_LABRAM_TRAINING = False
labram_metrics = {}
if RUN_LABRAM_TRAINING:
    labram_metrics = train_labram_model()
else:
    LOG.info("LaBraM training disabled; set RUN_LABRAM_TRAINING=True when ready.")
labram_metrics


In [ ]:

if LABRAM_METRICS_PATH.exists():
    pd.read_json(LABRAM_METRICS_PATH)
else:
    print("LaBraM metrics not available yet; run the training cell to generate them.")
